# Tutorial 5: Evaluation and Analysis

**Time**: 25 minutes

**Topics**:
- Stability metrics
- RDKit validation
- Uniqueness and novelty
- Property analysis
- Visualization

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
import numpy as np
import matplotlib.pyplot as plt
from qm9.analyze import analyze_stability_for_molecules, check_stability
from qm9.visualizer import visualize_molecule

print('Evaluation and Analysis Tutorial')

## 1. Load Generated Molecules

In [ ]:
# Example: Load from checkpoint
import pickle

# molecules = load_generated_molecules('outputs/my_model/samples.pkl')
# For demo, create synthetic
molecules = [
    {'positions': np.random.randn(8, 3), 'atom_types': np.array([1,1,1,1,0,0,0,0])}
    for _ in range(10)
]

print(f'Loaded {len(molecules)} molecules')

## 2. Stability Analysis

In [ ]:
from qm9.analyze import check_valency

dataset_info = {'atom_decoder': ['H', 'C', 'N', 'O', 'F']}

stability_results = []
for i, mol in enumerate(molecules):
    atom_types = mol['atom_types']
    charges = np.zeros(len(atom_types))
    
    valid = check_valency(atom_types, charges, dataset_info)
    stability_results.append(valid)
    
stability_rate = np.mean(stability_results)
print(f'Stability rate: {stability_rate:.2%}')

## 3. RDKit Validation

In [ ]:
try:
    from rdkit import Chem
    from qm9.analyze import build_molecule
    
    valid_mols = []
    for mol in molecules:
        try:
            rdkit_mol = build_molecule(mol['positions'], mol['atom_types'], dataset_info)
            if rdkit_mol is not None:
                Chem.SanitizeMol(rdkit_mol)
                valid_mols.append(rdkit_mol)
        except:
            pass
    
    validity = len(valid_mols) / len(molecules)
    print(f'Validity rate: {validity:.2%}')
    print(f'Valid molecules: {len(valid_mols)}/{len(molecules)}')
except ImportError:
    print('RDKit not installed - skipping validation')

## 4. Uniqueness

In [ ]:
try:
    smiles_set = set()
    for mol in valid_mols:
        smiles = Chem.MolToSmiles(mol)
        smiles_set.add(smiles)
    
    uniqueness = len(smiles_set) / len(valid_mols) if valid_mols else 0
    print(f'Uniqueness rate: {uniqueness:.2%}')
    print(f'Unique molecules: {len(smiles_set)}')
except:
    print('Uniqueness check requires RDKit')

## 5. Property Distribution

In [ ]:
# Example: Molecular weights
try:
    from rdkit.Chem import Descriptors
    
    weights = [Descriptors.MolWt(mol) for mol in valid_mols]
    
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.hist(weights, bins=20, alpha=0.7, edgecolor='black')
    plt.xlabel('Molecular Weight (u)')
    plt.ylabel('Count')
    plt.title('Generated Molecule Weights')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.boxplot(weights)
    plt.ylabel('Molecular Weight (u)')
    plt.title('Weight Distribution')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f'Weight range: [{min(weights):.1f}, {max(weights):.1f}]')
    print(f'Mean weight: {np.mean(weights):.1f} ± {np.std(weights):.1f}')
except:
    print('Property analysis requires RDKit')

## 6. Comprehensive Evaluation Script

```bash
python eval_analyze.py \\
    --model_path outputs/my_model \\
    --n_samples 10000 \\
    --save_molecules True
```

**Metrics computed**:
- Stability (atom/molecule level)
- Validity (RDKit sanitization)
- Uniqueness (SMILES dedupe)
- Novelty (vs training set)
- Property distributions

## Summary

✅ Stability analysis
✅ RDKit validation
✅ Uniqueness metrics
✅ Property distributions
✅ Comprehensive evaluation

### Quality Benchmarks:
- **Stability**: >95%
- **Validity**: >90%
- **Uniqueness**: >98%
- **Novelty**: >95%

Next: Tutorial 6 - Advanced Crystal Conditioning